# Flappy Bird: from Q-table to DQN 🐦

The sequel to [RL Snake - Q-Table to DQN](RL%20Snake%20-%20Q-Table%20to%20DQN.ipynb). Same plan – build the game, beat it with a Q-table, then with a DQN – but Flappy Bird adds three new difficulties:

- **Continuous state.** Height, speed and distances are real numbers, so before a table can be used *you* must decide how to chop them into boxes.
- **Bad random exploration.** Flapping at random sends the bird into the ceiling within a second. The Q-table here explores with *no* randomness at all.
- **Reward design.** Without random exploration, a single number – the price of dying – decides whether the Q-table learns at all.

| Milestone | What you build | You learn |
|---|---|---|
| 1 | The game as a headless environment + a random agent | physics in 10 lines; the RL interface again |
| 2 | Tabular Q-learning on a state grid you design | discretisation; how reward scale drives exploration |
| 3 | DQN in PyTorch on the raw continuous state | writing the DQN loss yourself; keeping the best snapshot |
| 4 | Race: random vs. Q-table vs. DQN + watch it fly | comparing agents fairly |

**How it works.** Cells marked `# TODO` are yours, each followed by a ✅/❌ check and a hidden solution (▶ *Solution*). The notebook runs top to bottom before you solve anything; the agents just won't learn.

**Runtime:** about 3–5 minutes on a laptop CPU once solved (good birds fly long games). **Needs:** `numpy`, `matplotlib`, `torch`.

Theory: [[RL Basics and MDPs]] · [[Q-Learning and Policy Gradients]].

In [ ]:
import numpy as np, matplotlib.pyplot as plt, copy, time
import torch, torch.nn as nn
from matplotlib import animation
from matplotlib.patches import Rectangle, Circle
from matplotlib.colors import ListedColormap
from IPython.display import HTML

torch.set_num_threads(1)   # tiny networks are fastest on one thread (and runs are reproducible)
def check(ok, msg_ok, msg_bad):
    print(("✅ " + msg_ok) if ok else ("❌ " + msg_bad))
def rolling(x, w=50):
    x = np.asarray(x, float); c = np.cumsum(np.insert(x, 0, 0))
    return np.array([(c[i + 1] - c[max(0, i + 1 - w)]) / min(i + 1, w) for i in range(len(x))])

---
## Milestone 1 – The game
**World.** The screen is 100 units high. The bird sits at x = 20 and only moves up and down; the pipes move left by 1 unit per frame. Each pipe is 8 wide with a 30-unit gap whose centre is random between 25 and 75. A new pipe follows every 45 units.

**Physics** (one frame per step). Action `1` = flap sets the vertical speed to +3.5; action `0` = do nothing lets gravity pull it down by 0.5 per frame, to at most −5. Then `y += v`.

**Rewards.** +0.1 for every frame alive, +1 for passing a pipe, and `death_reward` for crashing into a pipe, the floor or the ceiling. The game ends on a crash or after 50 pipes (so a perfect bird does not fly forever).

**State** – 5 numbers, scaled to roughly [−1, 1] for the neural network:

| # | Feature | Scale |
|---|---|---|
| 0 | bird height y | ÷ 100 |
| 1 | vertical speed v | ÷ 5 |
| 2 | horizontal distance to the next pipe | ÷ 45 |
| 3 | next gap centre − y (positive = gap is above) | ÷ 50 |
| 4 | the gap after that − y | ÷ 50 |

`state * Flappy.SCALE` gives the values back in game units.

In [ ]:
class Flappy:
    """Headless Flappy Bird. Actions: 0 = do nothing, 1 = flap."""
    H, X, R = 100.0, 20.0, 2.0                      # screen height, bird x, bird radius
    GAP, PIPE_W, SPACING, SPEED = 30.0, 8.0, 45.0, 1.0
    GRAVITY, FLAP_V, MAX_FALL = 0.5, 3.5, -5.0
    SCALE = np.array([H, 5.0, SPACING, 50.0, 50.0], np.float32)

    def __init__(self, death_reward=-1.0, max_score=50, seed=None):
        self.death_reward, self.max_score = death_reward, max_score
        self.rng = np.random.default_rng(seed)

    def reset(self):
        self.y, self.v = 50.0, 0.0
        self.pipes = [[self.X + 40 + i * self.SPACING, self.rng.uniform(25, 75)] for i in range(3)]   # [x, gap centre]
        self.score, self.steps = 0, 0
        return self.get_state()

    def step(self, action):
        self.v = self.FLAP_V if action == 1 else max(self.v - self.GRAVITY, self.MAX_FALL)
        self.y += self.v; self.steps += 1
        reward = 0.1
        for p in self.pipes:
            right_edge = p[0] + self.PIPE_W
            p[0] -= self.SPEED
            if right_edge >= self.X - self.R > p[0] + self.PIPE_W:     # the pipe just slid past the bird
                self.score += 1; reward = 1.0
        if self.pipes[0][0] + self.PIPE_W < 0:                           # recycle pipes that left the screen
            self.pipes.pop(0); self.pipes.append([self.pipes[-1][0] + self.SPACING, self.rng.uniform(25, 75)])
        crashed = not (self.R <= self.y <= self.H - self.R) or any(
            x - self.R < self.X < x + self.PIPE_W + self.R and abs(self.y - g) > self.GAP / 2 - self.R
            for x, g in self.pipes)
        if crashed:
            return self.get_state(), self.death_reward, True
        return self.get_state(), reward, self.score >= self.max_score

    def get_state(self):
        (x1, g1), (_, g2) = [p for p in self.pipes if p[0] + self.PIPE_W > self.X - self.R][:2]
        return np.array([self.y, self.v, x1 - self.X, g1 - self.y, g2 - self.y], np.float32) / self.SCALE

env = Flappy(seed=0)
s = env.reset()
print("start state (game units):", (s * Flappy.SCALE).round(1))
for t in range(12):
    s, r, done = env.step(1 if t % 6 == 0 else 0)   # flap every 6th frame
print("after 12 frames:", (s * Flappy.SCALE).round(1), "| reward", r, "| done", done)

In [ ]:
def draw(ax, y, pipes, score, title=""):
    ax.clear(); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_facecolor("#d0ebff")
    for x, g in pipes:
        ax.add_patch(Rectangle((x, 0), Flappy.PIPE_W, g - Flappy.GAP / 2, color="#2b8a3e"))
        ax.add_patch(Rectangle((x, g + Flappy.GAP / 2), Flappy.PIPE_W, 100, color="#2b8a3e"))
    ax.add_patch(Circle((Flappy.X, y), Flappy.R, color="#f59f00", ec="black"))
    ax.set_title(f"{title} score {score}")

fig, ax = plt.subplots(figsize=(4, 4)); draw(ax, env.y, env.pipes, env.score); plt.show()

### Baseline: a random agent
Every agent is evaluated the same way: **100 games with fixed pipes** (`seed=123`), always taking its best action, score = pipes passed (max 50).

In [ ]:
def play(env, policy):
    state, done = env.reset(), False
    while not done:
        state, _, done = env.step(policy(state))
    return env.score

def evaluate(policy, n=100, seed=123):
    env = Flappy(seed=seed)
    return np.array([play(env, policy) for _ in range(n)])

rng = np.random.default_rng(0)
random_scores = evaluate(lambda s: int(rng.integers(2)))
lengths = []
for _ in range(100):
    e = Flappy(seed=1); play(e, lambda s: int(rng.integers(2))); lengths.append(e.steps)
print(f"random agent: mean score {random_scores.mean():.2f}, survives {np.mean(lengths):.0f} frames on average")
print(f"the first pipe arrives after ~{40 - Flappy.R:.0f} frames")

The random bird never even reaches the first pipe: with a coin-flip every frame it flaps half the time and shoots into the ceiling. Keep that in mind: heavy random exploration (Snake's ε starting at 1) is wasted here – the bird would hardly ever see a pipe.

---
## Milestone 2 – Tabular Q-learning
### Exercise 1 – design the grid
A Q-table needs a finite set of states, so chop the continuous state into boxes. Write `discretize(state)` that returns a row number in `[0, N_STATES)` and set `N_STATES`. Tips:
- Feature 4 (the gap *after* next) barely matters for a table – leave it out.
- Too fine a grid → millions of rows that are each visited once and never learned. Too coarse → "just above the pipe edge" and "safely in the gap" fall in the same box. A couple of thousand rows is a good size.
- The speed takes only 18 different values (multiples of 0.5 from −5 to 3.5), so it can be used exactly.

In [ ]:
N_STATES = 1   # TODO: number of rows in your grid

def discretize(state):
    # TODO: map the 5-number state to a row number in [0, N_STATES)
    return 0

<details><summary>▶ Solution</summary>

```python
N_STATES = 6 * 17 * 18

def discretize(state):
    _, v, dx, dy, _ = state * Flappy.SCALE                 # back to game units
    dx_bin = int(np.clip(dx // 9, 0, 5))                    # distance to the pipe in 9-unit steps (0–5)
    dy_bin = int(np.clip(round(dy / 4), -8, 8)) + 8         # gap centre − y in 4-unit steps, clipped to ±32 (0–16)
    v_bin = int(round(v / 0.5)) + 10                        # speed is a multiple of 0.5 in [−5, 3.5] (0–17)
    return (dx_bin * 17 + dy_bin) * 18 + v_bin
```

Combine the bins like digits of a number: `(a * n_b + b) * n_c + c` is a unique row for every `(a, b, c)`. The exact bin widths are not critical – try 5 or 3 units and compare – but the vertical offset to the gap (`dy`) and the speed are what decides whether to flap.
</details>

In [ ]:
probe, srng = Flappy(seed=5), np.random.default_rng(5)
visited = []
for _ in range(30):                                       # states from a bird that flaps 1 frame in 7
    s, done = probe.reset(), False
    while not done:
        visited.append(s); s, _, done = probe.step(int(srng.random() < 1 / 7))
rows = np.array([discretize(s) for s in visited])
print(f"{len(visited)} probe states → {len(set(rows))} distinct rows, N_STATES = {N_STATES}")
check(rows.min() >= 0 and rows.max() < N_STATES and len(set(rows)) >= 50 and N_STATES <= 100_000,
      "Valid grid: rows in range and the states are actually told apart.",
      "Rows must lie in [0, N_STATES), distinguish at least 50 of the probe states, and N_STATES should stay ≤ 100 000.")

### Exercise 2 – what should dying cost?
We use **no random exploration** at all (ε = 0) and start the table at all zeros. Ties go to action 0 (glide), so the bird first tries gliding everywhere; it only tries flapping in a box once gliding there *looks worse than 0*.

The reward for staying alive is +0.1 per frame and passing a pipe is +1. The environment's default price of a crash is `death_reward = -1`. Run the training cell below with that value first, then come back and change `DEATH_PENALTY`. Which value lets the table learn?

In [ ]:
DEATH_PENALTY = -1.0   # ← TODO: experiment with this value (keep it negative)

<details><summary>▶ Solution</summary>

```python
DEATH_PENALTY = -10.0
```

With ε = 0 and a zero-initialised table, the agent switches action in a box only once the current action's Q-value drops **below 0**. Take a move that leads to a crash $k$ frames later. Its value is

$$V_k = 0.1\,\frac{1-\gamma^k}{1-\gamma} + \gamma^k D \qquad (\gamma = 0.99,\ D = \text{death reward}).$$

- $D = -1$: $V_k > 0$ for every $k \ge 10$. A crash ten or more frames away still *looks better than the untried alternative*, so the bird only reacts when it is about to hit something – usually too late.
- $D = -10$: $V_k < 0$ for every $k \le 68$, i.e. about 1.5 pipes ahead. Doomed moves are abandoned early enough to fix them.

Measured with this notebook's grid (100 test games, seeds 0–3): −1 → 2.7–5.5 pipes, −10 → 26–34 pipes.

Lesson: when exploration comes from the initial Q-values, the **scale** of the rewards decides how the agent explores. Snake has the same structure: +10 food, −10 death.
</details>

### Train the Q-table
The update rule is the one you wrote in the Snake notebook. Learning rate α = 0.5 (high, because the physics is deterministic – only the pipe gaps are random), discount γ = 0.99 (a flap pays off many frames later), 1500 games.

In [ ]:
def train_tabular(episodes=1500, alpha=0.5, gamma=0.99, seed=0):
    env = Flappy(death_reward=DEATH_PENALTY, seed=seed)
    Q, scores = np.zeros((N_STATES, 2)), []
    for ep in range(episodes):
        s, done = discretize(env.reset()), False
        while not done:
            a = int(np.argmax(Q[s]))                       # greedy; ties → 0 (glide)
            state2, r, done = env.step(a)
            s2 = discretize(state2)
            target = r if done else r + gamma * Q[s2].max()
            Q[s, a] += alpha * (target - Q[s, a])
            s = s2
        scores.append(env.score)
    return Q, scores

t0 = time.time()
Q, tab_curve = train_tabular()
tab_policy = lambda s: int(np.argmax(Q[discretize(s)]))
tab_scores = evaluate(tab_policy)
print(f"DEATH_PENALTY = {DEATH_PENALTY} | trained in {time.time() - t0:.0f}s | rows visited: {(Q != 0).any(1).sum()} of {N_STATES}")
print(f"Q-table agent: mean {tab_scores.mean():.2f} pipes ± {tab_scores.std():.2f}, {np.mean(tab_scores == 50):.0%} of games reach the cap of 50")
plt.plot(tab_curve, alpha=.25, label="pipes per game"); plt.plot(rolling(tab_curve), label="rolling mean (50)")
plt.xlabel("training game"); plt.ylabel("pipes"); plt.title("Tabular Q-learning"); plt.legend(); plt.show()
check(tab_scores.mean() >= 10, "The Q-table bird averages 10+ pipes.",
      "Below 10 pipes. Is Exercise 1 ✅? Then look hard at DEATH_PENALTY (Exercise 2).")

### What did the table learn?
Fix the distance to the next pipe at 20 units and ask the table, for every height offset and speed, what it would do – and *why*: was flapping ever tried in that box? Your discretisation decides the resolution of this picture.

In [ ]:
dys, vs = np.arange(-30, 31, 1.0), np.arange(-5, 3.6, 0.5)
def choice(dy, v):
    q = Q[discretize(np.array([50, v, 20, dy, 0], np.float32) / Flappy.SCALE)]
    if not q.any(): return 0                       # box never visited
    if q[1] > q[0]: return 3                       # flap
    return 2 if q[1] != 0 else 1                   # glide (flap tried and worse) / glide (flap never tried)
policy_map = np.array([[choice(dy, v) for v in vs] for dy in dys])
plt.imshow(policy_map, origin="lower", aspect="auto", cmap=ListedColormap(["#ced4da", "#ffc9c9", "#e03131", "#2f9e44"]),
           vmin=-.5, vmax=3.5, extent=[vs[0] - .25, vs[-1] + .25, dys[0] - .5, dys[-1] + .5], interpolation="nearest")
cb = plt.colorbar(ticks=range(4)); cb.ax.set_yticklabels(["never visited", "glide (flap untried)", "glide (flap worse)", "flap"])
plt.xlabel("vertical speed v"); plt.ylabel("gap centre − bird height"); plt.title("Q-table policy, pipe 20 units ahead")
plt.axhline(0, color="k", lw=.5); plt.show()

With the solution grid, most of the map is **pale red: the bird glides there and has never even tried flapping**. That is the zero-initialisation mechanism from Exercise 2 at work: gliding never looked worse than 0 in those boxes, so flapping never got a chance. The table found *a* policy that passes pipes, not the best one – and it never challenges it. Change the `20` in `choice` to look at other distances, and come back after the ε experiment in Milestone 4.

---
## Milestone 3 – DQN on the raw state
The network takes the 5 continuous numbers directly – no grid to design. Replay buffer, target network and linear ε-schedule are as in the Snake notebook; this time you write the **loss**.

In [ ]:
class ReplayBuffer:
    """Fixed-size ring buffer of transitions (s, a, r, s', done)."""
    def __init__(self, capacity, state_dim, rng):
        self.s, self.s2 = np.zeros((capacity, state_dim), np.float32), np.zeros((capacity, state_dim), np.float32)
        self.a, self.r, self.d = np.zeros(capacity, np.int64), np.zeros(capacity, np.float32), np.zeros(capacity, np.float32)
        self.capacity, self.size, self.pos, self.rng = capacity, 0, 0, rng
    def add(self, s, a, r, s2, done):
        i = self.pos
        self.s[i], self.a[i], self.r[i], self.s2[i], self.d[i] = s, a, r, s2, done
        self.pos = (i + 1) % self.capacity; self.size = min(self.size + 1, self.capacity)
    def sample(self, batch_size):
        j = self.rng.integers(self.size, size=batch_size)
        return tuple(torch.as_tensor(x[j]) for x in (self.s, self.a, self.r, self.s2, self.d))

def make_qnet(state_dim=5, n_actions=2, hidden=64):
    return nn.Sequential(nn.Linear(state_dim, hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_actions))

### Exercise 3 – the DQN loss
`batch = (s, a, r, s2, done)` as tensors of shapes `[B, 5]`, `[B]`, `[B]`, `[B, 5]`, `[B]`. Return the Huber loss (`nn.functional.smooth_l1_loss`) between

- the online network's Q-value **of the action taken**: $Q_\theta(s, a)$ – hint: `net(s).gather(1, a[:, None]).squeeze(1)`
- and the target $y = r + \gamma (1 - \text{done}) \max_{a'} Q_{\theta^-}(s', a')$ from the **target network**, computed without gradients.

In [ ]:
def dqn_loss(net, target_net, batch, gamma):
    s, a, r, s2, done = batch
    # TODO: return the Huber loss between Q(s, a) and the TD target
    return (net(s) * 0).sum()

<details><summary>▶ Solution</summary>

```python
def dqn_loss(net, target_net, batch, gamma):
    s, a, r, s2, done = batch
    q = net(s).gather(1, a[:, None]).squeeze(1)
    with torch.no_grad():
        y = r + gamma * (1 - done) * target_net(s2).max(dim=1).values
    return nn.functional.smooth_l1_loss(q, y)
```

Only $Q_\theta(s, a)$ for the action actually taken gets a gradient – the other action's output is untouched by this sample. Huber loss is quadratic for small errors and linear for large ones, so one surprising crash (a TD error of 10+) cannot blow up the gradient.
</details>

In [ ]:
torch.manual_seed(0)
n1, n2 = make_qnet(), make_qnet()
tb = (torch.randn(8, 5), torch.randint(0, 2, (8,)), torch.randn(8), torch.randn(8, 5), (torch.rand(8) < .3).float())
L = dqn_loss(n1, n2, tb, 0.99)
with torch.no_grad():
    want = nn.functional.smooth_l1_loss(n1(tb[0])[torch.arange(8), tb[1]], tb[2] + 0.99 * (1 - tb[4]) * n2(tb[3]).max(1).values)
L.backward()
check(torch.isclose(L, want) and n1[0].weight.grad.abs().sum() > 0 and n2[0].weight.grad is None,
      "Loss matches, gradients reach the online net only.",
      f"Loss {L.item():.4f}, expected {want.item():.4f}; gradients must flow into net but not into target_net.")

### Train the DQN
200 000 frames, one gradient step every 4 frames, target sync every 1000, ε from 1 to 0.01 over the first 5000 frames. Every 50 games the greedy policy is tested on 5 games and the best snapshot is kept: DQN performance can swing a lot from one evaluation to the next. A common explanation is *catastrophic forgetting*: once the buffer is full of good flights, the network rarely sees a crash and can unlearn what one looks like. This is the slowest cell in the notebook, 1–2 minutes.

In [ ]:
def train_dqn(total_steps=200_000, gamma=0.99, lr=1e-3, batch_size=64, eps_steps=5000, target_sync=1000,
              train_every=4, warmup=1000, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    env, test_env = Flappy(death_reward=DEATH_PENALTY, seed=seed), Flappy(seed=999)
    net, target = make_qnet(), make_qnet()
    target.load_state_dict(net.state_dict())
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    buf = ReplayBuffer(50_000, 5, rng)
    step, scores, tests, best = 0, [], [], (-1.0, None)
    while step < total_steps:
        s, done = env.reset(), False
        while not done:
            eps = max(0.01, 1 - step / eps_steps)
            if rng.random() < eps:
                a = int(rng.integers(2))
            else:
                with torch.no_grad():
                    a = int(net(torch.as_tensor(s)).argmax())
            s2, r, done = env.step(a)
            buf.add(s, a, r, s2, float(done)); s = s2; step += 1
            if buf.size >= warmup and step % train_every == 0:
                loss = dqn_loss(net, target, buf.sample(batch_size), gamma)
                opt.zero_grad(); loss.backward(); opt.step()
            if step % target_sync == 0:
                target.load_state_dict(net.state_dict())
        scores.append(env.score)
        if len(scores) % 50 == 0:
            with torch.no_grad():
                test = np.mean([play(test_env, lambda x: int(net(torch.as_tensor(x)).argmax())) for _ in range(5)])
            tests.append((len(scores), test))
            if test >= best[0]:
                best = (test, copy.deepcopy(net.state_dict()))
    net.load_state_dict(best[1])
    return net, scores, tests

t0 = time.time()
qnet, dqn_curve, dqn_tests = train_dqn()
def dqn_policy(s):
    with torch.no_grad():
        return int(qnet(torch.as_tensor(s)).argmax())
dqn_scores = evaluate(dqn_policy)
print(f"trained in {time.time() - t0:.0f}s on {len(dqn_curve)} games")
print(f"DQN agent: mean {dqn_scores.mean():.2f} pipes ± {dqn_scores.std():.2f}, {np.mean(dqn_scores == 50):.0%} of games reach the cap of 50")
check(dqn_scores.mean() >= 10, "The DQN bird averages 10+ pipes.",
      "Below 10 pipes – finish Exercise 3 (and set DEATH_PENALTY in Exercise 2: the DQN trains with it too).")

---
## Milestone 4 – The race

In [ ]:
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4))
a1.plot(rolling(tab_curve), label="Q-table"); a1.plot(rolling(dqn_curve), label="DQN")
a1.set_xlabel("training game"); a1.set_ylabel("pipes (rolling mean 50)"); a1.set_title("Learning curves (while exploring)"); a1.legend()
if dqn_tests:
    g, t = zip(*dqn_tests); a2.plot(g, t, "o-", color="C1", ms=3)
a2.set_xlabel("training game"); a2.set_ylabel("mean pipes, 5 greedy test games"); a2.set_title("DQN: greedy checkpoints")
names, results = ["random", "Q-table", "DQN"], [random_scores, tab_scores, dqn_scores]
a3.bar(names, [x.mean() for x in results], yerr=[x.std() for x in results], capsize=6, color=["#adb5bd", "#4c6ef5", "#e8590c"])
a3.set_ylabel("mean pipes over 100 test games (± std)"); a3.set_title("Greedy evaluation (cap 50)")
plt.tight_layout(); plt.show()
for n, x in zip(names, results):
    print(f"{n:8s} mean {x.mean():5.2f}  median {np.median(x):4.0f}  reached 50: {np.mean(x == 50):4.0%}")

**What to notice**
- The **DQN's training curve looks terrible** compared to its test score. With ε = 0.01 it still takes a random action once every 100 frames, and a random flap at the wrong moment is fatal. The table explores without randomness, so its training curve is honest.
- The **greedy checkpoints jump around** – a network that flew 50 pipes can crash on the first pipe 50 games later. That is why the training loop keeps the best snapshot instead of the last one.
- With the death penalty fixed, both learners go from 0 pipes to tens of pipes. With the solution code the DQN ended ahead on both seeds we tried (seed 0: Q-table 29.5 vs DQN 36.3 pipes; seed 1: 34.1 vs 50.0) – without any hand-designed boxes.

<details><summary>▶ Try it: give the Q-table a little randomness</summary>

In `train_tabular`, replace the `argmax` line with `a = int(rng.integers(2)) if rng.random() < EPS else int(np.argmax(Q[s]))` (create `rng = np.random.default_rng(seed)` at the top) and retrain with a constant `EPS`. Measured with the solution grid, 100 test games, seeds 0–3:

| death reward | ε = 0 | ε = 0.01 | ε = 0.1 |
|---|---|---|---|
| −1 | 2.7–5.5 | 34.0–48.5 | – |
| −10 | 26.3–34.1 | 11.3–50.0 | 4.0–39.9 |

A *little* randomness (1 % of frames) gets flapping tried in many more boxes – and then even −1 works, because exploration no longer depends on Q-values dropping below 0. Too much (10 %) causes crashes in states the bird would otherwise never reach, and results swing wildly between seeds. Exploration is a dial, not a switch.
</details>

### Watch it fly

In [ ]:
def record(policy, seed=7, max_frames=400):
    env = Flappy(seed=seed); s, done = env.reset(), False
    frames = [(env.y, [p[:] for p in env.pipes], env.score)]
    while not done and len(frames) < max_frames:
        s, _, done = env.step(policy(s)); frames.append((env.y, [p[:] for p in env.pipes], env.score))
    return frames

best_name, best_policy = max([("Q-table", tab_policy), ("DQN", dqn_policy)], key=lambda p: evaluate(p[1], 10).mean())
frames = record(best_policy)
fig, ax = plt.subplots(figsize=(4, 4))
anim = animation.FuncAnimation(fig, lambda i: draw(ax, *frames[i], title=f"{best_name} – frame {i},"), frames=len(frames), interval=30)
plt.close(fig)
print(f"{best_name}: {frames[-1][2]} pipes in the {len(frames) - 1} frames shown")
HTML(anim.to_jshtml())

---
## Go further
- **Harder game:** shrink `Flappy.GAP` to 24 or raise `Flappy.SPEED` to 1.5. Which agent copes better without retuning?
- **Coarser / finer grid:** halve and double your bin widths in `discretize`. Plot score against `N_STATES` – you should see a sweet spot.
- **ε-greedy for the table:** see the *Think about it* box above, then measure it.
- **Double DQN:** in `dqn_loss`, choose $a' = \arg\max_{a'} Q_\theta(s', a')$ with the online network and evaluate it with the target network.
- **Pixels:** render the screen to a small 20×20 image and train a CNN-DQN on it. Now there is no hand-made state at all – this is what the original Atari DQN did.
- Watch a Q-table learn Flappy Bird live in [RL Arcade.html](RL%20Arcade.html) – and try to beat it yourself.